<a href="https://colab.research.google.com/github/Atharv2200/GroupDNA-WhatsApp-Analytics/blob/main/GroupDna_Atharv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 GroupDNA: WhatsApp Group Chat Analyzer

Atharv Mandekar

In [ ]:
import os
from datetime import datetime, timedelta
import numpy as np

def get_count(item):
    return item[1]

FILE_PATH = "hostel_bois.txt"

def is_new_message(line):
    if len(line) > 8:
        if line[2] == '/' and line[5] == '/':
            return True
    return False

with open(FILE_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    lines = f.readlines()

messages = []
system_count = 0
media_count = 0
deleted_count = 0
current_msg = None

for raw_line in lines:
    line = raw_line.strip()
    if line == "":
        continue

    if is_new_message(line):
        if current_msg != None:
            messages.append(current_msg)

        parts = line.split(" - ", 1)

        if len(parts) < 2:
            parts = line.split(" ", 2)
            if len(parts) >= 3 and "," in parts[0]:
                ts_part = parts[0] + " " + parts[1]
                ts_part = ts_part.strip()
                rest = parts[2]
            else:
                system_count = system_count + 1
                current_msg = None
                continue
        else:
            ts_part = parts[0]
            rest = parts[1]

        if ":" in rest:
            sender_and_content = rest.split(":", 1)
            sender = sender_and_content[0].strip()
            content = sender_and_content[1].strip()

            is_media = False
            if "<Media omitted>" in content:
                is_media = True
                media_count = media_count + 1

            is_deleted = False
            if "This message was deleted" in content or "You deleted this message" in content:
                is_deleted = True
                deleted_count = deleted_count + 1

            current_msg = {
                "timestamp_str": ts_part,
                "sender": sender,
                "text": content,
                "is_media": is_media,
                "is_deleted": is_deleted
            }
        else:
            system_count = system_count + 1
            current_msg = None
    else:
        if current_msg != None:
            current_msg["text"] = current_msg["text"] + " " + line

if current_msg != None:
    messages.append(current_msg)

In [ ]:
def parse_date(ts_str):
    ts_str = ts_str.replace('\u202f', ' ').strip()
    formats = ["%d/%m/%y, %H:%M", "%d/%m/%Y, %H:%M", "%d/%m/%y, %I:%M %p", "%d/%m/%Y, %I:%M %p"]

    for fmt in formats:
        try:
            return datetime.strptime(ts_str, fmt)
        except ValueError:
            continue
    return None

person_counts = {}
day_counts = {}
hour_counts = {}
valid_dates = []

for msg in messages:
    sender = msg["sender"]

    if sender in person_counts:
        person_counts[sender] = person_counts[sender] + 1
    else:
        person_counts[sender] = 1

    dt = parse_date(msg["timestamp_str"])
    if dt != None:
        msg["dt"] = dt
        valid_dates.append(dt)

        day_str = dt.strftime("%d %B %Y")
        if day_str in day_counts:
            day_counts[day_str] = day_counts[day_str] + 1
        else:
            day_counts[day_str] = 1

        hour_str = dt.strftime("%H:00")
        if hour_str in hour_counts:
            hour_counts[hour_str] = hour_counts[hour_str] + 1
        else:
            hour_counts[hour_str] = 1

valid_dates.sort()
start_date = valid_dates[0]
end_date = valid_dates[-1]
total_days = (end_date.date() - start_date.date()).days + 1
if total_days < 1:
    total_days = 1

sorted_participants = sorted(person_counts.items(), key=get_count, reverse=True)

busiest_day = ("N/A", 0)
if len(day_counts) > 0:
    busiest_day = max(day_counts.items(), key=get_count)

busiest_hour = ("N/A", 0)
if len(hour_counts) > 0:
    busiest_hour = max(hour_counts.items(), key=get_count)

In [ ]:
STOP_WORDS = ['i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'that', 'this', 'you', 'me', 'my', 'we', 'us', 'hai', 'ki', 'ko', 'se', 'ke', 'ka', 'ye', 'wo', 'to']

word_freq = {}
punctuations = "!()-[]{};:'\"\,<>./?@#$%^&*_~"

for msg in messages:
    if msg["is_media"] == True or msg["is_deleted"] == True:
        continue

    clean_text = ""
    for char in msg["text"].lower():
        if char not in punctuations:
            clean_text = clean_text + char
        else:
            clean_text = clean_text + " "

    words = clean_text.split()
    for w in words:
        if w != "" and w not in STOP_WORDS and len(w) > 2:
            if w in word_freq:
                word_freq[w] = word_freq[w] + 1
            else:
                word_freq[w] = 1

top_words = sorted(word_freq.items(), key=get_count, reverse=True)[:10]